# Sayf-Eval analysis walkthrough

A guided, runnable tour of the analysis behind *"Benchmark Scores Are
Pipeline-Dependent: A Reliability Audit of Cybersecurity LLM Benchmarks."*

Each notebook **reuses** the `analysis/` modules (no logic is copied) and renders
the already-computed artifacts in `analysis/reports/`. Light steps recompute
live; heavy steps (GPU/API) are rendered from cache unless you opt in.

## How to use
- Run cells top-to-bottom. Light steps recompute in seconds–minutes; if
  `outputs/` isn't mounted they fall back to the cached `reports/` artifact.
- **Heavy steps are off by default.** To recompute embeddings (GPU) or the
  verification / K-A judges (Azure API), launch Jupyter with `SAYF_NB_REGEN=1`.
- Kernel: any Python ≥3.10 with the analysis deps + `requirements-notebooks.txt`.

## The measurement-pipeline framing
Every benchmark is modeled as a 5-stage pipeline — **Dataset 𝒟 · Prompt 𝒫 ·
Inference 𝓘 · Extraction/scoring 𝓔 · Aggregation 𝒜** — and a reported score is
conditional on the whole pipeline, not an intrinsic model property. The analysis
quantifies 15 recurring failure modes across these stages.

## Notebook map
| Notebook | Theme | Failure modes |
|---|---|---|
| `01_results_table` | Master accuracy table + secondary metrics (F1, MAD) | 𝒜 aggregation |
| `02_judge_agreement` | LLM-judge reliability (Cohen's κ across judges) | 𝓔 extraction/scoring |
| `03_gold_errors_verification` | Suspect gold labels + search-grounded verification | 𝐹₂(𝒟) label quality |
| `04_capability_coverage` | Knowledge-vs-Analytical coverage | 𝐹₁(𝒟) limited coverage |
| `05_redundancy_correlation_embeddings` | Cross-task redundancy, effective dimensions | benchmark redundancy |


In [ ]:
import nbtools as nb
from nbtools import show_df, show_fig, show_md, run_mod, heavy, REGEN, REPORTS_DIR
print("SAYF_NB_REGEN =", REGEN, "  (heavy GPU/API steps run only when True)")
print("reports dir  :", REPORTS_DIR)

### What's already computed (`reports/` inventory)

In [ ]:
for p in sorted(REPORTS_DIR.iterdir()):
    print(("dir  " if p.is_dir() else "file ") + p.name)

### Run order
`results_table` produces `per_model_task_accuracy.json`, consumed by
`gold_error_voting` and `correlation`. Otherwise the notebooks are independent;
`00`→`05` is a sensible reading order.
